# MMSD concat3 comparison baseline

Matches `concat3_paper/concat3_padded_train.ipynb` except for ImageNet-only image features and run identifiers. Existing seed artifacts and summary.json in concat3_result belong to the previous implementation; rerun this notebook for updated results.

In [1]:
%pip install tqdm==4.67.1


  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Configuration


In [2]:
import ast
import csv
import json
import os
import pickle
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
FEATURE_DIR = DATA_ROOT / 'features/resnetv2_50_a1h_in1k'
ATTRIBUTE_PREDICTIONS = Path('/kaggle/input/datasets/fruich/resnet101-coco/resnet101_coco_attributes/extract_all')
SEEDS = (42, 43, 44)
BATCH_SIZE = 32
EPOCHS = 8
PATIENCE = 5
LEARNING_RATE = 0.001
L2 = 1e-7
SEQ_LEN = 75
NUM_WORKERS = 4
RUN_ROOT = Path('/kaggle/working/runs/concat3_imagenet_own_attributes_packed')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


PyTorch: 2.11.0+cu130 Device: cuda:0


## Data and embeddings


In [3]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        vocabulary = VocabularyLoader(stream, encoding='latin1').load()
    if not isinstance(vocabulary, dict) or not all(
        isinstance(key, str) and type(value) is int for key, value in vocabulary.items()):
        raise ValueError(f'Invalid vocabulary: {path}')
    return vocabulary

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
attribute_vocab = load_vocabulary(ASSET_ROOT / 'ExtractWords/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
attribute_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'ExtractWords/vector', dtype=np.float32))
assert word_weights.shape == (len(word_vocab), 200)
assert attribute_weights.shape == (len(attribute_vocab), 200)
assert word_vocab['<pad>'] == 0
attribute_rows = [ast.literal_eval(line) for line in
                  ATTRIBUTE_PREDICTIONS.read_text().splitlines() if line.strip()]
attributes = {str(row[0]): row[1:] for row in attribute_rows}
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}

class SarcasmDataset(Dataset):
    def __init__(self, split):
        rows = [ast.literal_eval(line) for line in
                (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
        self.rows = [row for row in rows if not blocked.intersection(row[1].split())]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.attributes = torch.empty((len(self.rows), 5), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for index, row in enumerate(self.rows):
            ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[index, :len(ids)] = torch.tensor(ids, dtype=torch.long)
            self.attributes[index] = torch.tensor([attribute_vocab.get(token, attribute_vocab['<unk>'])
                                                   for token in attributes[str(row[0])]], dtype=torch.long)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        tweet_id = str(self.rows[index][0])
        image = torch.from_numpy(np.load(FEATURE_DIR / f'{tweet_id}.npy', allow_pickle=False))
        return self.tokens[index], image, self.attributes[index], self.labels[index], tweet_id

train_data = SarcasmDataset('train')
valid_data = SarcasmDataset('valid2')
test_data = SarcasmDataset('test2')
generator = torch.Generator().manual_seed(SEEDS[0])
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', generator=generator,
                          persistent_workers=NUM_WORKERS > 0)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
positives = train_data.labels.sum().item()
positive_weight = (len(train_data) - positives) / positives
print('Train / validation / test:', len(train_data), len(valid_data), len(test_data))
print('Positive-class loss weight:', positive_weight)


Train / validation / test: 19816 2410 2409
Positive-class loss weight: 1.292987734320759


## Concat3 model


In [4]:
class Concat3Classifier(nn.Module):
    def __init__(self, word_weights, attribute_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False, padding_idx=0)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.image_projection = nn.Linear(2048, 1024)
        self.attribute_embedding = nn.Embedding.from_pretrained(attribute_weights.clone(), freeze=False)
        self.attribute_attention = nn.Sequential(nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 1))
        self.classifier = nn.Sequential(nn.Linear(1736, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)
        nn.init.xavier_uniform_(self.image_projection.weight)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens, images, attributes):
        lengths = tokens.ne(0).sum(dim=1)
        if (lengths == 0).any():
            raise ValueError('Encountered a text sequence containing only padding')
        embedded = self.dropout(self.word_embedding(tokens))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        text_raw, _ = pad_packed_sequence(packed_output, batch_first=True,
                                          total_length=tokens.size(1))
        mask = (torch.arange(tokens.size(1), device=tokens.device)[None, :]
                < lengths[:, None]).unsqueeze(-1)
        text_vector = (text_raw * mask).sum(dim=1) / lengths.unsqueeze(1)
        image_vector = self.image_projection(images).relu().mean(dim=1)
        attribute_raw = self.attribute_embedding(attributes)
        weights = self.attribute_attention(attribute_raw).softmax(dim=1)
        attribute_vector = (weights * attribute_raw).sum(dim=1)
        fused = torch.cat([image_vector, text_vector, attribute_vector], dim=1)
        return self.classifier(fused).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')


## Evaluation


In [5]:
def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    predicted = np.asarray(probabilities) > 0.5
    tp = int(((labels == 1) & predicted).sum())
    fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum())
    fn = int(((labels == 1) & ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / len(labels), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}

@torch.no_grad()
def evaluate(network, loader):
    network.eval()
    labels, probabilities, tweet_ids = [], [], []
    for tokens, images, attributes, targets, ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = network(tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True),
                         attributes.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist())
        tweet_ids.extend(ids)
    return binary_metrics(labels, probabilities), tweet_ids, labels, probabilities


## Training


In [6]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
all_results = []
for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    train_loader.generator.manual_seed(seed)
    run_dir = RUN_ROOT / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=False)
    feature_manifests = [json.loads(path.read_text()) for path in sorted((FEATURE_DIR / 'manifests').glob('*.json'))]
    config = {'model': 'concat3', 'variant': 'imagenet_features_own_attributes_packed', 'attribute_predictions': str(ATTRIBUTE_PREDICTIONS), 'attribute_embedding': 'trainable', 'seed': seed, 'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS,
              'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'lr_decay_per_epoch': 0.8,
              'l2': L2, 'sequence_length': SEQ_LEN, 'text_padding': 'packed_lstm_masked_mean', 'hidden_size': 256, 'input_dropout': 0.2,
              'positive_weight': positive_weight, 'checkpoint_selection': 'validation positive-class F1',
              'asset_root': str(ASSET_ROOT), 'feature_dir': str(FEATURE_DIR),
              'split_counts': {'train': len(train_data), 'valid': len(valid_data), 'test': len(test_data)},
              'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
              'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
              'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
              'gpu_names': [torch.cuda.get_device_name(DEVICE)] if DEVICE.type == 'cuda' else [],
              'device': str(DEVICE), 'num_workers': NUM_WORKERS,
              'persistent_workers': NUM_WORKERS > 0, 'non_blocking_transfers': True,
              'feature_manifests': feature_manifests}
    (run_dir / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
    base_model = Concat3Classifier(word_weights, attribute_weights).to(DEVICE)
    network = base_model
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.Adam([p for p in base_model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
    best_f1, stale_epochs = -1.0, 0
    history = []
    start = time.perf_counter()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    for epoch in range(1, EPOCHS + 1):
        network.train()
        epoch_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', unit='batch')
        for tokens, images, attributes, labels, _ in progress:
            tokens, images = tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True)
            attributes, labels = attributes.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(network(tokens, images, attributes), labels) + L2 * base_model.regularization()
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(base_model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += loss.item() * len(labels)
            seen += len(labels)
            progress.set_postfix(loss=epoch_loss / seen)
        valid_metrics, _, _, _ = evaluate(network, valid_loader)
        record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
                  'learning_rate': optimizer.param_groups[0]['lr'],
                  'seconds': time.perf_counter() - epoch_start}
        history.append(record)
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
        print(json.dumps(record))
        if valid_metrics['f1'] > best_f1:
            best_f1, stale_epochs = valid_metrics['f1'], 0
            temporary = run_dir / 'best.pt.tmp'
            torch.save({'model': base_model.state_dict(), 'epoch': epoch, 'validation_f1': best_f1}, temporary)
            os.replace(temporary, run_dir / 'best.pt')
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE:
            break
    training_seconds = time.perf_counter() - start
    peak_memory = [torch.cuda.max_memory_allocated(DEVICE)] if DEVICE.type == 'cuda' else []
    print('Training seconds:', training_seconds, 'Best validation F1:', best_f1)
    checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE, weights_only=True)
    base_model.load_state_dict(checkpoint['model'])
    test_start = time.perf_counter()
    test_metrics, ids, labels, probabilities = evaluate(network, test_loader)
    test_seconds = time.perf_counter() - test_start
    results = {'model': 'concat3', 'seed': seed, 'selected_epoch': checkpoint['epoch'],
               'best_validation_f1': checkpoint['validation_f1'], 'test': test_metrics,
               'training_seconds_including_validation': training_seconds,
               'test_seconds_including_data_loading': test_seconds,
               'peak_allocated_gpu_bytes_per_device': peak_memory,
               'trainable_parameters': sum(p.numel() for p in base_model.parameters() if p.requires_grad)}
    (run_dir / 'results.json').write_text(json.dumps(results, indent=2) + '\n')
    with (run_dir / 'test_predictions.csv').open('w', newline='') as stream:
        writer = csv.writer(stream)
        writer.writerow(['tweet_id', 'label', 'probability', 'prediction'])
        writer.writerows((tweet_id, int(label), probability, int(probability > 0.5))
                         for tweet_id, label, probability in zip(ids, labels, probabilities))
    print(json.dumps(results, indent=2))
    all_results.append(results)
    del network, base_model, optimizer, scheduler, criterion
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
summary = {metric: {'mean': float(np.mean([r['test'][metric] for r in all_results])),
                    'std': float(np.std([r['test'][metric] for r in all_results], ddof=1))}
           for metric in ('precision', 'recall', 'f1', 'accuracy')}
summary['seeds'] = all_results
(RUN_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))


Epoch 1/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 1, "train_loss": 0.43290118719052423, "validation": {"precision": 0.8222996515679443, "recall": 0.7382690302398331, "f1": 0.778021978021978, "accuracy": 0.8323651452282158, "tp": 708, "fp": 153, "tn": 1298, "fn": 251}, "learning_rate": 0.001, "seconds": 458.432200672}


Epoch 2/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 2, "train_loss": 0.267107535069372, "validation": {"precision": 0.7679924242424242, "recall": 0.8456725755995829, "f1": 0.8049627791563275, "accuracy": 0.8369294605809129, "tp": 811, "fp": 245, "tn": 1206, "fn": 148}, "learning_rate": 0.0008, "seconds": 145.79469736199997}


... (62 outputs left out)

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 7, "train_loss": 0.019949303086665014, "validation": {"precision": 0.7709335899903753, "recall": 0.835245046923879, "f1": 0.8018018018018018, "accuracy": 0.8356846473029046, "tp": 801, "fp": 238, "tn": 1213, "fn": 158}, "learning_rate": 0.0002621440000000001, "seconds": 99.55089403700003}
Training seconds: 748.6644131170001 Best validation F1: 0.8076923076923077


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{
  "model": "concat3",
  "seed": 44,
  "selected_epoch": 2,
  "best_validation_f1": 0.8076923076923077,
  "test": {
    "precision": 0.7913741223671013,
    "recall": 0.8227320125130344,
    "f1": 0.8067484662576687,
    "accuracy": 0.8430884184308842,
    "tp": 789,
    "fp": 208,
    "tn": 1242,
    "fn": 170
  },
  "training_seconds_including_validation": 748.6644131170001,
  "test_seconds_including_data_loading": 23.11181120299989,
  "peak_allocated_gpu_bytes_per_device": [
    308857344
  ],
  "trainable_parameters": 6157690
}
{
  "precision": {
    "mean": 0.7695830879308904,
    "std": 0.018997315241060492
  },
  "recall": {
    "mean": 0.8369829683698297,
    "std": 0.013203628719654192
  },
  "f1": {
    "mean": 0.8016545457679256,
    "std": 0.004500824709487284
  },
  "accuracy": {
    "mean": 0.8350629583506296,
    "std": 0.006950252306558434
  },
  "seeds": [
    {
      "model": "concat3",
      "seed": 42,
      "selected_epoch": 2,
      "best_validation_f1": 0.804962

## Final test and saved results
